# OpenRouter Model Selection

This notebook demonstrates how to query available models, filter by capabilities, and compare providers.

**Requires:** `OPENROUTER_API_KEY` environment variable.

Uses the free model `qwen/qwen3.6-plus:free` for completion examples. Free models have strict rate limits — the helper function retries automatically.

In [ ]:
import os
import time
import requests

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY environment variable"

BASE_URL = "https://openrouter.ai/api/v1"
HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}


def openrouter_chat(model, messages, retries=3, delay=10):
    """Chat completion with retry on rate limit."""
    for attempt in range(retries):
        response = requests.post(
            f"{BASE_URL}/chat/completions",
            headers=HEADERS,
            json={"model": model, "messages": messages},
        )
        if response.status_code == 429:
            wait = delay * (attempt + 1)
            print(f"Rate limited, waiting {wait}s... (attempt {attempt+1}/{retries})")
            time.sleep(wait)
            continue
        response.raise_for_status()
        return response.json()
    response.raise_for_status()


print("API key configured.")

## List Available Models

In [ ]:
response = requests.get(f"{BASE_URL}/models", headers=HEADERS)
response.raise_for_status()
models = response.json()["data"]

print(f"Total models available: {len(models)}")
print("\nFirst 10 models:")
for m in models[:10]:
    print(f"  {m['id']}")

## Filter Free Models

In [ ]:
free_models = [
    m for m in models
    if m.get("pricing", {}).get("prompt", "1") == "0"
    and m.get("pricing", {}).get("completion", "1") == "0"
]

print(f"Free models: {len(free_models)}")
for m in free_models:
    ctx = m.get("context_length", "?")
    print(f"  {m['id']:50s}  context: {ctx}")

## Inspect a Specific Model

In [ ]:
import json

# Find our target model
target = "qwen/qwen3.6-plus:free"
model_info = next((m for m in models if m["id"] == target), None)

if model_info:
    print(json.dumps(model_info, indent=2))
else:
    print(f"Model {target} not found in model list.")
    base = target.split(":")[0]
    matches = [m for m in models if m["id"].startswith(base)]
    if matches:
        print(f"\nFound related models:")
        for m in matches:
            print(f"  {m['id']}")

## Compare Responses Across Models

Send the same prompt to multiple free models and compare outputs.
Free models have strict rate limits — this cell spaces requests with delays.

In [ ]:
test_models = [m["id"] for m in free_models[:3]]
prompt = "Explain quantum entanglement in exactly two sentences."

for model_id in test_models:
    try:
        data = openrouter_chat(
            model_id,
            [{"role": "user", "content": prompt}],
            retries=3,
            delay=15,
        )
        answer = data["choices"][0]["message"]["content"]
        tokens = data.get("usage", {}).get("total_tokens", "?")
        print(f"\n--- {model_id} ({tokens} tokens) ---")
        print(answer)
    except Exception as e:
        print(f"\n--- {model_id} ---")
        print(f"Error: {e}")
    time.sleep(15)  # space out requests for free tier